In [1]:
import os
import json
import random
import time

BASE_DIR = r"E:\Praxis\TERM2\DAS\retail_parquet"
MODELS_DIR = os.path.join(BASE_DIR, "mlops_pipeline", "models")
DRIFT_LOG_PATH = os.path.join(MODELS_DIR, "drift_log.json")

def simulate_monitoring():
    """
    Simulates a monitoring service that tracks incoming data for distribution shifts (data drift).
    In a real scenario, this would compare live data distributions vs training data distributions.
    """
    print("Starting drift monitoring simulation...")
    
    # Simulate baseline (from training)
    baseline_stats = {
        "avg_freight": 22.8,
        "avg_weight": 2100.0,
        "late_delivery_rate": 0.08
    }
    
    print(f"Baseline Stats: {baseline_stats}")
    print("Monitoring incoming batches...")
    
    for batch_id in range(1, 6):
        time.sleep(1)
        # Simulate incoming batch statistics with slight random variations
        current_freight = baseline_stats["avg_freight"] + random.uniform(-2, 5)
        current_weight = baseline_stats["avg_weight"] + random.uniform(-200, 300)
        current_rate = baseline_stats["late_delivery_rate"] + random.uniform(-0.02, 0.05)
        
        drift_detected = False
        alerts = []
        
        if current_freight > baseline_stats["avg_freight"] * 1.15:
            drift_detected = True
            alerts.append(f"Freight cost drift detected (+{((current_freight/baseline_stats['avg_freight'])-1)*100:.1f}%)")
            
        if current_rate > baseline_stats["late_delivery_rate"] * 1.3:
            drift_detected = True
            alerts.append(f"Late delivery rate spike detected! Currently at {current_rate*100:.1f}%")
            
        log_entry = {
            "batch_id": batch_id,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "stats": {
                "avg_freight": current_freight,
                "avg_weight": current_weight,
                "late_delivery_rate": current_rate
            },
            "drift_detected": drift_detected,
            "alerts": alerts
        }
        
        print(f"Batch {batch_id} processed. Drift Detected: {drift_detected}")
        if drift_detected:
            for alert in alerts:
                print(f"  -> ALERT: {alert}")
                
        # Append to log
        logs = []
        if os.path.exists(DRIFT_LOG_PATH):
            with open(DRIFT_LOG_PATH, "r") as f:
                logs = json.load(f)
        logs.append(log_entry)
        with open(DRIFT_LOG_PATH, "w") as f:
            json.dump(logs, f, indent=4)

if __name__ == "__main__":
    simulate_monitoring()



Starting drift monitoring simulation...
Baseline Stats: {'avg_freight': 22.8, 'avg_weight': 2100.0, 'late_delivery_rate': 0.08}
Monitoring incoming batches...
Batch 1 processed. Drift Detected: True
  -> ALERT: Late delivery rate spike detected! Currently at 12.2%
Batch 2 processed. Drift Detected: True
  -> ALERT: Late delivery rate spike detected! Currently at 12.7%
Batch 3 processed. Drift Detected: False
Batch 4 processed. Drift Detected: False
Batch 5 processed. Drift Detected: False
